# EnerGIS Framework - Runner

Haupteinstiegspunkt für Optimierungsläufe mit dem EnerGIS Planning Framework.

## Übersicht

Dieses Notebook führt einen vollständigen Optimierungslauf durch:
- **Perfect Forecast (PF)**: Optimale Dimensionierung über den gesamten Zeitraum
- **Rolling Horizon (RH)**: Operative Planung mit rollendem Horizont
- **Model Predictive Control (MPC)**: RH mit Forecast-Updates
- **PF → RH/MPC**: Kombinierter Workflow mit Design-Fixierung

## Quick Start

1. Alle Zellen mit **Run All** ausführen
2. Bei Bedarf Config-Pfade in Zelle 3 anpassen
3. Ergebnisse werden automatisch in `saved_workflows/` gespeichert
4. Dashboard wird optional am Ende angezeigt

---

## 1. Setup & Imports

In [ ]:
# Bootstrap: Projekt-Root finden und zum Pfad hinzufügen
import sys
from pathlib import Path

# Finde Projekt-Root (suche nach .git und energis/ Verzeichnis)
current = Path.cwd()
project_root = None

for candidate in [current] + list(current.parents):
    if (candidate / '.git').exists() and (candidate / 'energis').exists():
        project_root = candidate
        break

if project_root is None:
    # Fallback: Suche nach Projektname
    for candidate in [current] + list(current.parents):
        if candidate.name == 'Planing-Framework-for-Heat':
            project_root = candidate
            break

if project_root is None:
    project_root = current

# Füge zum Python-Pfad hinzu (damit energis importiert werden kann)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Jetzt können wir notebook_helpers importieren
try:
    from energis.io.notebook_helpers import setup_notebook_environment
    
    PROJECT_ROOT = setup_notebook_environment()
    print("\n✅ Setup abgeschlossen")
    
except ImportError as e:
    print(f"❌ Import-Fehler: {e}")
    print("\n💡 Fehlende Dependencies installieren:")
    print("   cd " + str(project_root))
    print("   pip install pandas numpy openpyxl matplotlib pyomo")
    print("\n   Oder mit allen Notebook-Dependencies:")
    print("   pip install -e .[notebooks]")
    raise

In [ ]:
# Imports
from datetime import datetime
from energis.run import rolling_horizon as rh
from energis.io.notebook_helpers import (
    save_workflow_run,
    display_workflow_summary,
    display_kpi_summary,
    create_and_display_dashboard
)

print("✅ Imports erfolgreich")

## 2. Konfiguration

Die Konfiguration erfolgt über YAML-Dateien, die in der angegebenen Reihenfolge gemerged werden.
Spätere Dateien überschreiben frühere Einträge.

### Standard-Konfiguration:
- `base.yaml` - Basis-Einstellungen (Solver, Zeitschritt, etc.)
- `tech_catalog.yaml` - Technologie-Katalog (Komponenten-Definitionen)
- `default.site.yaml` - Standort-Daten (Input-Daten, Zeitzone, etc.)
- `baseline.system.yaml` - System-Topologie (Komponenten, Kapazitäten)
- `pf_then_rh.workflow.scenario.yaml` - Szenario (Run-Mode, RH-Parameter)

Passe die Config-Pfade nach Bedarf an!

In [ ]:
# Konfigurationsdateien
CONFIG_PATHS = [
    'configs/base.yaml',
    'configs/tech_catalog.yaml',
    'configs/sites/default.site.yaml',
    'configs/systems/baseline.system.yaml',
    'configs/scenarios/pf_then_rh.workflow.scenario.yaml',
]

# Optional: Overrides für spezifische Parameter
# Beispiele:
# - Run-Mode ändern: {'scenario': {'run_mode': 'PF_ONLY'}}
# - Solver ändern: {'run': {'solver': 'glpk'}}
# - RH-Parameter: {'scenario': {'rolling_horizon': {'heat_horizon_hours': 72}}}
OVERRIDES = None

# Config-Dateien prüfen
print("📋 Konfigurationsdateien:")
all_exist = True
for cfg_path in CONFIG_PATHS:
    full_path = PROJECT_ROOT / cfg_path
    exists = full_path.exists()
    symbol = '✅' if exists else '❌'
    print(f"  {symbol} {cfg_path}")
    if not exists:
        all_exist = False

if not all_exist:
    raise FileNotFoundError("Nicht alle Config-Dateien gefunden!")

print("\n✅ Konfiguration OK")

## 3. Workflow ausführen

Der Workflow führt die Optimierung gemäß der konfigurierten Run-Mode aus:

- **PF_ONLY**: Nur Perfect Forecast
- **RH_ONLY**: Nur Rolling Horizon
- **MPC_ONLY**: Nur Model Predictive Control (mit Forecasts)
- **PF_THEN_RH**: PF für Dimensionierung, dann RH mit fixiertem Design
- **PF_THEN_MPC**: PF für Dimensionierung, dann MPC mit fixiertem Design

In [ ]:
%%time
print("="*70)
print("🚀 STARTE OPTIMIERUNG")
print("="*70)
print(f"Start: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

try:
    workflow = rh.run_workflow(CONFIG_PATHS, overrides=OVERRIDES)
    
    print("\n" + "="*70)
    print("✅ OPTIMIERUNG ERFOLGREICH")
    print("="*70)
    print(f"\n📊 Workflow: {' → '.join(workflow.plan.steps)}")
    
    optimization_success = True
    
except Exception as e:
    print("\n" + "="*70)
    print("❌ FEHLER")
    print("="*70)
    print(f"\nFehler: {e}\n")
    
    import traceback
    traceback.print_exc()
    
    workflow = None
    optimization_success = False

## 4. Workflow speichern & exportieren

Speichert den Workflow mit allen Ergebnissen, Metadaten und Plots in `saved_workflows/`.

In [ ]:
if optimization_success and workflow:
    # Workflow-Name und Beschreibung (anpassbar)
    WORKFLOW_NAME = "Baseline Simulation"
    WORKFLOW_DESCRIPTION = "PF + RH Optimierung mit Standard-Konfiguration"
    
    # Workflow speichern (inkl. CSV, PDF, SVG Exports)
    workflow_dir = save_workflow_run(
        workflow,
        name=WORKFLOW_NAME,
        description=WORKFLOW_DESCRIPTION,
        config_paths=CONFIG_PATHS
    )
    
    print(f"\n💡 Dashboard anzeigen:")
    print(f"   • In diesem Notebook: Siehe Zelle 7")
    print(f"   • In interactive_dashboard.ipynb: Workflow auswählen")
    
else:
    print("⚠️  Workflow-Speicherung übersprungen (Optimierung fehlgeschlagen)")

## 5. Zusammenfassung

Zeigt die wichtigsten Kennzahlen aus dem Optimierungslauf.

In [ ]:
if optimization_success and workflow:
    # Zusammenfassung anzeigen
    display_workflow_summary(workflow)
else:
    print("⚠️  Keine Ergebnisse verfügbar")

## 6. Key Performance Indicators

Detaillierte KPI-Analyse mit Kostenaufschlüsselung und Komponentenauslastung.

In [ ]:
if optimization_success and workflow:
    # Detaillierte KPI-Analyse
    display_kpi_summary(workflow)
else:
    print("⚠️  Keine KPIs verfügbar")

## 7. Interaktives Dashboard (Optional)

Zeigt ein interaktives Dashboard mit allen Visualisierungen.

**Hinweis:** Dashboard benötigt Panel, Holoviews und Plotly.

Installation: `pip install panel holoviews bokeh plotly`

In [ ]:
# Dashboard aktivieren/deaktivieren
SHOW_DASHBOARD = True  # Auf False setzen um Dashboard zu überspringen

if optimization_success and workflow and SHOW_DASHBOARD:
    try:
        dashboard = create_and_display_dashboard(
            workflow,
            title=f"Runner - {WORKFLOW_NAME}"
        )
        
        # Dashboard anzeigen
        dashboard
        
    except ImportError as e:
        print(f"❌ Dashboard-Import-Fehler: {e}")
        print("   Installation: pip install panel holoviews bokeh plotly")
elif not SHOW_DASHBOARD:
    print("ℹ️  Dashboard deaktiviert (SHOW_DASHBOARD = False)")
    print("   Zum Aktivieren: SHOW_DASHBOARD = True setzen")
else:
    print("⚠️  Dashboard kann nicht erstellt werden (keine Ergebnisse)")

---

## 📚 Weitere Informationen

- **Dokumentation**: `README.md`, `ARCHITECTURE_V2.md`
- **Methodologie**: `docs/methodology.md`
- **CLI-Nutzung**: `python -m energis.run.rolling_horizon --help`
- **Andere Notebooks**:
  - `interactive_dashboard.ipynb` - Dashboard mit gespeicherten Workflows
  - `scenario_studio.ipynb` - Interaktive Szenario-Analyse

## 🌐 Dashboard als Webapp

Um das Dashboard als eigenständige Webapp zu starten:

```bash
panel serve runner.ipynb --show
# Oder auf spezifischem Port:
panel serve runner.ipynb --port 5006 --show
```

---